In [3]:
import numpy as np
import pandas as pd
import sqlite3

df = pd.read_csv('netflix_real_time_project_dataset.csv')

print(f"Total Rows: {df.shape[0]} | Total Columns: {df.shape[1]}")
df.head(3)

Total Rows: 2500 | Total Columns: 14


,show_id,title,type,genre,country,language,release_year,date_added,rating,runtime_minutes,seasons,imdb_score,imdb_votes,tmdb_popularity
0,N00001,Elite 1,Movie,Documentary,Japan,English,2000,2024-04-23,G,78,NaN,7.2,682269,8.79
1,N00002,Extraction,Movie,Documentary,United States,Hindi,2018,2020-07-04,TV-PG,102,NaN,8.4,49394,24.93
2,N00003,Dark 3,TV Show,Documentary,United States,English,2019,2025-04-25,PG,59,6.0,6.9,120803,50.57


In [8]:
# 1. Date conversion & Time features
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month_name()

# 2. Missing values & Handling Nulls
df['country'] = df['country'].fillna('Unknown')
df['rating'] = df['rating'].fillna('Not Rated')
df['language'] = df['language'].fillna('Unknown')

# 3. NumPy conditional logic: Clean runtime and seasons
# Movie-ku seasons NaN aagum, TV show-ku runtime_minutes NaN aagum
df['runtime_minutes'] = np.where(df['type'] == 'Movie', df['runtime_minutes'], np.nan)
df['seasons'] = np.where(df['type'] == 'TV Show', df['seasons'], np.nan)

# 4. Content rating category using NumPy select
conditions = [
    df['rating'].isin(['G', 'TV-G', 'TV-Y', 'TV-Y7']),
    df['rating'].isin(['PG', 'TV-PG', 'PG-13', 'TV-14']),
    df['rating'].isin(['R', 'TV-MA', 'NC-17'])
]
categories = ['Kids/Family', 'Teens/Adult-Lite', 'Mature/Adults']
df['audience_category'] = np.select(conditions, categories, default='General')

print("Data Cleaning & Feature Engineering Completed!")

Data Cleaning & Feature Engineering Completed!


In [5]:
# Cleaned DataFrame-ah SQLite table-ah save panrom
conn = sqlite3.connect('netflix.db')
df.to_sql('netflix_catalog', conn, if_exists='replace', index=False)

# Colab-la SQL run panna extension enable panrom
%load_ext sql
%sql sqlite:///netflix.db

In [7]:
# Function to run SQL queries cleanly
def run_sql(query):
  return pd.read_sql_query(query, conn)

In [10]:
q1 = """
SELECT
    COUNT(show_id) AS total_titles,
    SUM(CASE WHEN type = 'Movie' THEN 1 ELSE 0 END) AS total_movies,
    SUM(CASE WHEN type = 'TV Show' THEN 1 ELSE 0 END) AS total_tv_shows,
    ROUND(AVG(imdb_score), 2) AS avg_imdb_score,
    COUNT(DISTINCT country) AS total_countries
FROM netflix_catalog;
"""
run_sql(q1)

,total_titles,total_movies,total_tv_shows,avg_imdb_score,total_countries
0,2500,1730,770,7.01,10


In [11]:
q2 = """
SELECT
    country,
    COUNT(show_id) AS total_content,
    ROUND(AVG(imdb_score), 2) AS avg_imdb
FROM netflix_catalog
GROUP BY country
ORDER BY total_content DESC
LIMIT 10;
"""
run_sql(q2)

,country,total_content,avg_imdb
0,United Kingdom,282,6.93
1,Japan,274,7.04
2,Spain,254,7.03
3,Canada,252,7.03
4,United States,250,7.00
5,India,247,7.03
6,South Korea,242,6.92
7,France,238,7.02
8,Germany,233,7.07
9,Australia,228,7.07


In [12]:
q3 = """
WITH yearly_data AS (
    SELECT
        release_year,
        COUNT(show_id) AS total_releases
    FROM netflix_catalog
    GROUP BY release_year
)
SELECT
    release_year,
    total_releases,
    LAG(total_releases, 1) OVER (ORDER BY release_year) AS prev_year_releases,
    ROUND(
        (total_releases - LAG(total_releases, 1) OVER (ORDER BY release_year)) * 100.0 /
        NULLIF(LAG(total_releases, 1) OVER (ORDER BY release_year), 0),
        2
    ) AS yoy_growth_pct
FROM yearly_data
ORDER BY release_year;
"""
run_sql(q3)

,release_year,total_releases,prev_year_releases,yoy_growth_pct
0,2000,95,NaN,NaN
1,2001,105,95.0,10.53
2,2002,95,105.0,-9.52
3,2003,96,95.0,1.05
4,2004,83,96.0,-13.54
5,2005,95,83.0,14.46
6,2006,95,95.0,0.00
7,2007,90,95.0,-5.26
8,2008,96,90.0,6.67
9,2009,100,96.0,4.17
